## Model Analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/distributed_graph_flow/blob/main/doc/docs/tutorial/model_analysis.ipynb)

GraphFlow model analysis is an automated tool to understand and interpret a
trained model. For example, it helps to debug model quality, to guide the
development of new input features, and to detect label leakage.

In this tutorial, you will load the MAG citation graph, train a node prediction
model to predict the publication venue (`labels`) of each paper, run
`model.analyze()` to inspect the model, and interpret the results.

## Installing GF

Make sure your machine has a GPU or a TPU, otherwise training will take forever.
On Google Colab, you can get one for free: Go to "Edit > Notebook settings" and
select a GPU or TPU hardware accelerator.

In [ ]:
# Install DGF (Distributed Graph Flow) and OGB (for the toy dataset).
!pip install dgf ogb -U

## Importing libraries

In [2]:
import dgf  # Import Graph Flow

## Getting the graph data

We first fetch the MAG graph: a heterogeneous graph of papers, authors,
institutions, and fields of study.

In [3]:
# Download the MAG graph from the OGB repository.
graph, schema = dgf.io.fetch_ogb_graph("mag")

Caching mag graph at /tmp/gf_fetch/mag.cache
OGB dependency not available. Downloading graph from CNS.


The `#split` feature of the `paper` nodeset indicates the OGB train / validation
/ test split of each paper. It is not a property of the papers, so we remove it
from the schema to make sure the model does not use it as an input feature.

In [4]:
del schema.node_sets["paper"].features["#split"]

## Training a model

We train a Graph Convolutional Network (GCN) to predict the `labels` feature of
the `paper` nodeset, i.e., the venue (conference or journal) where the paper was
published.

In [5]:
model = dgf.learning.train_node_model(
    graph=graph,
    schema=schema,
    target_nodeset="paper",
    target_column="labels",
    architecture="gcn",
)

Using gpu JAX backend
Graph input schema:
Node Sets:
  author:
    | Feature   | Format   | Semantic   | Shape   | Detail   |
    |-----------|----------|------------|---------|----------|
    | #id       | BYTES    | PRIMARY_ID | None    |          |

  field_of_study:
    | Feature   | Format   | Semantic   | Shape   | Detail   |
    |-----------|----------|------------|---------|----------|
    | #id       | BYTES    | PRIMARY_ID | None    |          |

  institution:
    | Feature   | Format   | Semantic   | Shape   | Detail   |
    |-----------|----------|------------|---------|----------|
    | #id       | BYTES    | PRIMARY_ID | None    |          |

  paper:
    | Feature   | Format     | Semantic    | Shape   | Detail   |
    |-----------|------------|-------------|---------|----------|
    | #id       | BYTES      | PRIMARY_ID  | None    |          |
    | feat      | FLOAT_32   | EMBEDDING   | (128,)  |          |
    | labels    | INTEGER_64 | CATEGORICAL | None    |       

[Warning] Validation set truncated from 73638 to 32000 nodes due to caching limits. The extra nodes are used for training instead. To use more validation nodes, either increase `num_valid_steps` or set `cache_valid_dataset=False`.


Num. training seed nodes: 704389, Num. validation seed nodes: 32000
Create graph sampler
Compute feature statistics
  GraphFeatureStatistics:
  Node Sets (4):
    'author':
      '#id': count=450403
    'field_of_study':
      '#id': count=544390
    'institution':
      '#id': count=39360
    'paper':
      '#id': count=2969242
      'feat': count=2969242
      'labels': count=9984, min=0.0000, max=348.0000
      'year': count=2969242, min=2010.0000, max=2019.0000, quantiles=(100)[2010.0000, 2010.0000, 2010.0000, ..., 2019.0000, 2019.0000, 2019.0000]

Compute graph statistics for padding
  padding: Node Sets:
  author: 2604 nodes
  field_of_study: 2517 nodes
  institution: 291 nodes
  paper: 13659 nodes

Edge Sets:
  affiliated_with: 447 edges
  cites: 9550 edges
  has_topic: 11213 edges
  writes: 5228 edges


[Warning] No normalizer created for node set 'author', feature '#id'.
[Warning] No normalizer created for node set 'paper', feature '#id'.
[Warning] No normalizer created for node set 'field_of_study', feature '#id'.
[Warning] No normalizer created for node set 'institution', feature '#id'.


Preparing dataset finished in 5.34 seconds
Normalizer:
Graph Normalizer:

Node Sets:
  author:
    (No normalizers)

  field_of_study:
    (No normalizers)

  institution:
    (No normalizers)

  paper:
    - feat: IdentityNormalizer
    - labels: IdentityNormalizer
    - year: SoftQuantileNormalizer

Edge Sets:
  affiliated_with: (Source: author, Target: institution)
    (No normalizers)

  cites: (Source: paper, Target: paper)
    (No normalizers)

  has_topic: (Source: paper, Target: field_of_study)
    (No normalizers)

  writes: (Source: author, Target: paper)
    (No normalizers)

Normalized graph schema:
Node Sets:
  author:
    (No features)

  field_of_study:
    (No features)

  institution:
    (No features)

  paper:
    | Feature            | Format     | Semantic    | Shape   | Detail       |
    |--------------------|------------|-------------|---------|--------------|
    | feat               | FLOAT_32   | EMBEDDING   | (128,)  |              |
    | labels            

...


[Warning] Skipped 64 out of 320192 training samples due to insufficient padding. Consider increasing `padding_margin`.
[Warning] The model has likely not converged: Training stopped at the maximum number of training steps (`num_train_steps`=10000) while the validation loss was still improving. The model quality can likely be improved by increasing `num_train_steps` or `learning_rate`.


Final model evaluation
Evaluating model on generator


Evaluation:   0%|          | 0/10000 [00:00<?, ?it/s]

...Tracing model


Evaluation: 100%|██████████| 10000/10000 [00:08<00:00, 1203.53it/s]


Before analyzing a model, always look at its description (e.g., evaluation
metrics, training logs, hyper-parameters). An analysis of a poorly trained model
is of little value.

In [6]:
model.describe()

Type,Node prediction model - Predict the value of a node feature.
Target nodeset,paper
Target column,labels
Number of label classes,349
WARNING,"Validation set truncated from 73638 to 32000 nodes due to caching limits. The extra nodes are used for training instead. To use more validation nodes, either increase `num_valid_steps` or set `cache_valid_dataset=False`."
WARNING,"No normalizer created for node set 'author', feature '#id'."
WARNING,"No normalizer created for node set 'paper', feature '#id'."
WARNING,"No normalizer created for node set 'field_of_study', feature '#id'."
WARNING,"No normalizer created for node set 'institution', feature '#id'."
WARNING,Skipped 64 out of 320192 training samples due to insufficient padding. Consider increasing `padding_margin`.
WARNING,The model has likely not converged: Training stopped at the maximum number of training steps (`num_train_steps`=10000) while the validation loss was still improving. The model quality can likely be improved by increasing `num_train_steps` or `learning_rate`.


## Analyzing the model

Once the model is trained, we analyze it with `model.analyze(graph)`. The
analysis perturbs each component of the graph (feature, edgeset, nodeset) one at
a time, and measures how much the model quality degrades. See the "Glossary" tab
for details.

In [7]:
model.analyze(graph)

...Tracing model
Analyzing model on 512 seed nodes across 14 perturbations
...Tracing model


Analyzing model: 100%|██████████| 14/14 [00:37<00:00,  2.67s/it]


ModelAnalysis(
  target=paper.labels, task=NODE_CLASSIFICATION, baseline_accuracy=0.3594, baseline_loss=2.4431, num_seed_nodes=512, edge_perturbation_mode=MASLOV_SNEPPEN, duration=44.65s
  [Features]
     1. paper.feat                   delta_loss=+3.4965, delta_accuracy=+0.3359, grad_saliency=138.2241
     2. paper.year                   delta_loss=+0.0376, delta_accuracy=+0.0137, grad_saliency=1.7110
     3. field_of_study.#id           delta_loss=-0.0000, delta_accuracy=+0.0000
     4. paper.#id                    delta_loss=-0.0000, delta_accuracy=+0.0000
     5. institution.#id              delta_loss=-0.0000, delta_accuracy=+0.0000
     6. author.#id                   delta_loss=-0.0000, delta_accuracy=+0.0000
  [Edgesets]
     1. writes                       delta_loss=+0.5167, delta_accuracy=+0.0762
     2. cites                        delta_loss=+0.4733, delta_accuracy=+0.1133
     3. has_topic                    delta_loss=+0.2508, delta_accuracy=+0.0449
     4. affiliated_with              delta_loss=+0.0039, delta_accuracy=+0.0059
  [Nodesets]
     1. paper                        delta_loss=+3.5868, delta_accuracy=+0.3359
     2. author                       delta_loss=+0.5142, delta_accuracy=+0.0898
     3. field_of_study               delta_loss=+0.2506, delta_accuracy=+0.0566
     4. institution                  delta_loss=+0.0035, delta_accuracy=+0.0059
)

The "Component importance" tab shows the importance of each feature, edgeset,
and nodeset, perturbed one at a time. Looking at the components in descending order of importance, we can make some observations:

-   **paper.feat**: The `feat` feature of the `paper` nodeset is by far the most
    important feature. This makes sense: This feature encodes the text of the
    paper, which is very indicative of the venue where it was published.
-   **writes** and **author**: The authors of a paper (i.e., the `author`
    nodeset and the `writes` edgeset) are important to predict its venue: After
    all, an author probably publishes mostly in the same venues. Note that both
    components are equally important, which also makes sense: The model needs
    both of them to know about the authors.
-   **cites**: The papers cited by a paper (i.e., the `cites` edgeset) are also
    informative: After all, papers tend to cite papers from the same venues.
-   **field_of_study**: Similarly, the fields of study of a paper (i.e., the
    `field_of_study` nodeset and the `has_topic` edgeset) are informative: Each
    venue covers a specific set of topics.
-   **year**: The publication `year` of a paper has a small effect on the
    prediction: Venues come and go, and their popularity changes over time.
-   **institution** and **affiliated_with**: Interestingly, institutions (i.e.,
    the `institution` nodeset and the `affiliated_with` edgeset) barely matter.
-   **#id**: The `#id` features have zero importance. This makes sense, as the
    model does not use them as input features at all (default in GraphFlow).

## Conclusion

In this tutorial, you trained a node prediction model and used `model.analyze()`
to find which components of the graph the model relies on. Use the same approach
on your own models to debug their quality, to check that new input features are
actually used, and to detect suspiciously important components that could
indicate label leakage.